# Lezione 18: Mettere in sicurezza gli agenti AI con ricevute crittografiche

## Notebook pratico

Questo notebook guida attraverso quattro esercizi:

1. **Firma la tua prima ricevuta** per una chiamata a uno strumento agente e verifica la firma.
2. **Manometti la ricevuta** e osserva il fallimento della verifica.
3. **Costruisci una catena di tre ricevute** e conferma l'integrità della catena.
4. **Incorpora una chiamata a uno strumento del Microsoft Agent Framework** in modo che ogni azione generi una ricevuta.

Tutte le primitive crittografiche sono importate da librerie ben mantenute (`pynacl` per Ed25519, `jcs` per JSON canonico RFC 8785, `hashlib` dalla libreria standard Python per SHA-256). La logica della ricevuta stessa è Python semplice che puoi leggere e modificare.

Esegui le celle in ordine. Ogni sezione è breve e autonoma.


## Setup

Installa le due dipendenze. Entrambe hanno licenze permissive (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Utility di supporto

Questi due helper gestiscono la codifica base64url (senza padding) e l'hashing SHA-256 di oggetti arbitrari. Mantengono il resto del notebook focalizzato sulla logica della ricevuta stessa.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Firma la tua prima ricevuta

Immagina che il nostro agente per **Contoso Travel** abbia appena cercato voli da Sydney a Los Angeles per un cliente. Vogliamo registrare questa chiamata allo strumento come una ricevuta firmata affinché un revisore futuro possa verificarla senza doverci fidare.

### Step 1.1: Genera una chiave di firma

In produzione, la chiave di firma dell'agente risiederebbe in un modulo di sicurezza hardware (HSM), Azure Key Vault, o in un archivio protetto simile. Per questa lezione generiamo una nuova chiave in memoria.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: Creare il payload della ricevuta

Il payload contiene tutto ciò a cui vogliamo che la ricevuta attesti: chi ha agito, quale strumento, con quali argomenti, cosa è stato restituito, sotto quale politica e quando. Calcoliamo l’hash degli argomenti e del risultato invece di includerli direttamente affinché la ricevuta non divulghi contenuti sensibili.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: Firma e assemblaggio della ricevuta

Tre passaggi:

1. Canonicalizzare il payload utilizzando JCS in modo che due implementazioni che producono la stessa ricevuta logica generino byte identici.
2. Hashare i byte canonicalizzati con SHA-256.
3. Firmare l'hash con la chiave privata Ed25519.

La firma viene quindi allegata al payload originale per produrre la ricevuta finale.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: Verifica della ricevuta

La verifica inverte il processo. Rimuoviamo la firma, ricalcoliamo l’hash canonico e controlliamo la firma rispetto alla chiave pubblica nella ricevuta.

Un revisore che effettua questa verifica non ha bisogno di nulla da noi tranne la ricevuta stessa. Nessun servizio da chiamare, nessuna directory di chiavi da interrogare, nessuna fiducia richiesta.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Dovresti vedere `Receipt is valid: True`. L'agente ha prodotto il suo primo record di audit firmato crittograficamente.


## Sezione 2: Manomettere la ricevuta

Lo scopo delle ricevute è che siano evidenti in caso di manomissione. Dimostriamolo.

Modificheremo esattamente un carattere della ricevuta e osserveremo la mancata verifica.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Cosa è appena successo?

Quando abbiamo modificato `policy_id`, i byte canonici sono cambiati. L'hash SHA-256 di quei byte è cambiato. La firma (che era basata sull'hash originale) non corrisponde più al nuovo hash. La verifica restituisce correttamente `False`.

Non è possibile modificare alcun campo della ricevuta e farla comunque verificare, a meno che l'attaccante non abbia la chiave privata. Finché la chiave privata è in un key vault e la chiave pubblica è pubblicata, è impossibile nascondere manomissioni.

Provalo tu stesso: modifica `tool_name` o `agent_id` o `timestamp` nella cella sopra e riesegui. Ogni modifica produce una ricevuta non valida.


## Sezione 3: Collegare le ricevute insieme

Una singola ricevuta protegge una singola azione. La maggior parte degli agenti compie molte azioni. Per rendere l'intera sequenza evidenziabile in caso di manomissione, colleghiamo ogni ricevuta a quella precedente includendo l'hash della ricevuta precedente nel payload della nuova ricevuta.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Se qualcuno rimuove o riordina una ricevuta, la catena si interrompe esattamente in quel punto. La verifica di qualsiasi ricevuta successiva fallisce perché il suo `previous_receipt_hash` non corrisponde più all'hash effettivo del suo predecessore.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Ora interrompi la catena manomettendo la ricevuta centrale e verifica nuovamente. La ricevuta manomessa non supera il controllo della firma, E la ricevuta successiva non supera il controllo del collegamento nella catena (poiché il suo `previous_receipt_hash` non corrisponde più all'hash della ricevuta centrale modificata).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

La ricevuta 0 verifica ancora (non è stata modificata e non ha un predecessore da cui dipendere). La ricevuta 1 non supera il controllo della firma perché abbiamo modificato `tool_args_hash`. La ricevuta 2 non supera il controllo del collegamento a catena perché il suo `previous_receipt_hash` è stato calcolato sulla ricevuta originale 1 (ora modificata).

Anche se un attaccante riesce a ri-firmare la ricevuta 1 modificata (cosa che non può fare senza la chiave privata), la discrepanza del collegamento a catena nella ricevuta 2 esporrebbe comunque la manomissione. Per nascondere la modifica, l’attaccante dovrebbe ri-firmare ogni ricevuta dal punto di modifica in poi, il che richiede il possesso della chiave privata.


## Sezione 4: Avvolgere una chiamata a uno strumento agente con firma della ricevuta

In un'implementazione reale, non vuoi che ogni autore di agenti ricordi di chiamare `make_receipt`. Vuoi che la firma della ricevuta sia automatica per ogni invocazione dello strumento.

Ecco il modello più semplice: una classe wrapper che prende qualsiasi funzione strumento richiamabile e restituisce una versione di essa che emette una ricevuta. Questo si adatta a qualsiasi framework per agenti, incluso il Microsoft Agent Framework (`agent_framework.azure`).

Se non hai configurato un progetto Azure AI Foundry, il mock locale qui sotto dimostra comunque il modello.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integrazione con il Microsoft Agent Framework

Il wrapper `ReceiptedTool` sopra è indipendente dal framework. Per usarlo all'interno di un agent costruito con il Microsoft Agent Framework, registra la funzione incapsulata come uno strumento. Uno schizzo (sostituiresti il mock con una vera registrazione dello strumento Azure AI Foundry):

```python
# Pseudodice che mostra la forma dell'integrazione.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Sei un agente di viaggio Contoso ...",
#     tools=[receipted_lookup],   # lo strumento incapsulato, non la funzione grezza
# )
# response = agent.run("Trova voli da Sydney a Los Angeles a giugno.")
#
# # Dopo l'esecuzione, ogni chiamata allo strumento effettuata dall'agente ha una ricevuta firmata:
# audit_chain = receipted_lookup.receipts
```

Il framework dell'agente non deve sapere nulla sulle ricevute. La firma della ricevuta è incapsulata intorno allo strumento, non integrata nel framework. Questo è il modo per aggiungere provenienza al codice dell'agente esistente senza riscriverlo.


## Riepilogo e sfida extra

Hai:

- Generato una coppia di chiavi Ed25519.
- Costruito e firmato una ricevuta per una chiamata a uno strumento agent.
- Verificato la ricevuta offline usando solo la chiave pubblica.
- Manomesso una ricevuta e osservato il fallimento della verifica.
- Costruito una sequenza concatenata tramite hash di tre ricevute.
- Manomesso il mezzo della catena e osservato sia il fallimento della firma che il fallimento del collegamento nella catena.
- Incapsulato una funzione dello strumento con la firma automatica delle ricevute.

**Sfida extra.** Estendi lo schema della ricevuta con un campo `request_id` (un UUID per il tracciamento distribuito). Aggiorna `make_receipt` per includerlo e conferma che le ricevute si verifichino ancora da un estremo all'altro. Quindi modifica il campo dopo la firma e conferma che la verifica fallisce. Questo ti obbliga a interiorizzare come ogni byte della codifica canonica contribuisce alla firma.

**Confine importante.** Le ricevute dimostrano tre cose e solo tre cose: attribuzione (questa chiave ha firmato questo contenuto), integrità (il contenuto non è cambiato dalla firma) e ordinamento (questa ricevuta è arrivata dopo quella ricevuta). NON dimostrano che l'azione dell'agente sia stata corretta, che la policy indicata in `policy_id` sia stata effettivamente valutata, o che l'agente abbia seguito ogni regola. Le ricevute sono una base. Il governo è il sistema che costruisci sopra.

Rileggi il README della lezione con questo confine in mente. L'errore più comune che i team commettono con le ricevute è presumere che "abbiamo le ricevute" significhi "siamo governati". Non è così. Le ricevute rendono il comportamento degli agenti verificabile. Non lo rendono corretto.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
